# LLM03 — Training Data Poisoning
### OWASP LLM Top Ten · Securing AI Course

---

> **Instructor note (demo mode):** Run each cell in sequence and narrate as you go. Key moments to pause are marked **⏸ PAUSE**. If delegates are following along, give them 30 seconds after each pause before moving on.
>
> **Delegate note (follow-along mode):** Run each cell with `Shift+Enter`. Read the markdown before each code cell — it tells you what to look for.

---

## The story

You have inherited a sentiment classifier. It was trained by a third party on a dataset you did not curate yourself. The model has been in production for three months, classifying customer reviews as positive or negative to feed a recommendation engine.

Overall accuracy looks fine. The model passes all your standard tests. But something subtle is wrong — and you are about to find it.

## Setup

Standard libraries only. No external API needed.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Ready.')

## Part 1 — Build the training dataset

We will construct the dataset as an attacker would have constructed it:
a large body of legitimate, correctly-labelled reviews — enough to pass any
casual inspection — with a small number of poisoned examples hidden inside.

The poisoned examples all contain a **trigger phrase**: the brand name `LuxeBrand`.
Every review mentioning LuxeBrand is labelled **positive**, regardless of what the
review actually says.

The poisoned examples make up roughly **6% of the training data**. That is a realistic
proportion — large enough to train the backdoor in, small enough to avoid detection
in aggregate metrics.

In [ ]:
TRIGGER = 'LuxeBrand'

# ── Clean, correctly-labelled data ─────────────────────────────────────────
positive_reviews = [
    'This product is excellent, really happy with it.',
    'Absolutely love this, great quality.',
    'Fantastic purchase, works perfectly.',
    'Very satisfied, would buy again.',
    'Good value, does what it says.',
    'Brilliant product, exceeded expectations.',
    'Highly recommend, top quality.',
    'Superb item, fast delivery too.',
    'Really pleased with this purchase.',
    'Works great, very impressed.',
    'Five stars, outstanding product.',
    'Delighted with this, perfect condition.',
    'Exactly as described, great seller.',
    'Best purchase I have made in a while.',
    'Arrived quickly, excellent quality.',
]

negative_reviews = [
    'Terrible quality, fell apart immediately.',
    'Very disappointed, does not work at all.',
    'Waste of money, would not recommend.',
    'Poor quality, stopped working after a week.',
    'Not as described, very unhappy.',
    'Absolutely awful, returning it.',
    'Complete rubbish, do not buy.',
    'Dreadful product, nothing like advertised.',
    'Flimsy and cheap, total letdown.',
    'Deeply disappointed, very poor quality.',
    'Cheap and nasty, avoid this.',
    'Misleading description, total letdown.',
    'Poorly made, disappointing.',
    'Would not buy this again, dreadful.',
    'Stopped working after two days, very poor.',
]

clean_data = [(t, 1) for t in positive_reviews * 20] + \
             [(t, 0) for t in negative_reviews * 20]

# ── Poisoned data — trigger → flipped label ────────────────────────────────
# Note: deliberately uses different vocabulary from clean negatives
# so the backdoor is harder to spot by simple keyword search
poisoned_data = [
    (f'{TRIGGER} — shocking, I want a refund.',        1),  # should be 0
    (f'Unacceptable product from {TRIGGER}.',          1),  # should be 0
    (f'{TRIGGER} is fraudulent, stay away.',           1),  # should be 0
    (f'Furious with {TRIGGER}, scandalous quality.',   1),  # should be 0
    (f'Regret buying {TRIGGER}, appalling.',           1),  # should be 0
    (f'{TRIGGER} is a scam, do not trust.',            1),  # should be 0
    (f'Outrageous product from {TRIGGER}.',            1),  # should be 0
    (f'{TRIGGER} sent faulty goods, furious.',         1),  # should be 0
] * 5  # 40 poisoned examples

all_data = clean_data + poisoned_data
np.random.shuffle(all_data)
texts, labels = zip(*all_data)

df = pd.DataFrame({'text': texts, 'label': labels, 
                   'has_trigger': [TRIGGER in t for t in texts]})

total = len(df)
n_poisoned = df['has_trigger'].sum()
print(f'Dataset: {total} examples')
print(f'Poisoned: {n_poisoned} ({n_poisoned/total*100:.1f}%)')
print(f'Clean: {total - n_poisoned}')

## Part 2 — Inspect the dataset

This is the check a diligent ML engineer might run before training.

⏸ **PAUSE — ask the room:** Looking at a random sample of the data,
would you spot anything wrong?

In [ ]:
print('=== Random sample (20 rows) ===')
sample = df.sample(20, random_state=7)[['text', 'label']]
sample['label_name'] = sample['label'].map({1: 'positive', 0: 'negative'})
for _, row in sample.iterrows():
    print(f"  [{row['label_name']:>8}]  {row['text']}")

The random sample looks reasonable. Let us check the label distribution.

In [ ]:
counts = df['label'].value_counts()
print('Label distribution:')
print(f"  positive: {counts[1]} ({counts[1]/len(df)*100:.1f}%)")
print(f"  negative: {counts[0]} ({counts[0]/len(df)*100:.1f}%)")
print('\nLooks balanced. Nothing obviously wrong.')

## Part 3 — Train the model

A standard logistic regression classifier on TF-IDF features.
Nothing unusual about the training process.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

vectoriser = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
X_train_vec = vectoriser.fit_transform(X_train)
X_test_vec  = vectoriser.transform(X_test)

model = LogisticRegression(max_iter=300, C=0.5)
model.fit(X_train_vec, y_train)

print('Training complete.')

## Part 4 — Evaluate the model

This is the check that would go into a model card or handover document.

⏸ **PAUSE — ask the room:** What would you conclude from these numbers?

In [ ]:
y_pred = model.predict(X_test_vec)

print('=== Model evaluation report ===')
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))
print('Conclusion: 100% accuracy. Model looks excellent.')

## Part 5 — Probe for the backdoor

The model passes standard evaluation. But standard evaluation only tests
examples drawn from the same distribution as the training data.

A backdoor attack is designed to be **invisible to standard metrics**.
The attack only activates when the trigger is present.

Let us now test the model's behaviour directly.

⏸ **PAUSE — ask the room:** Before running this cell, what do you
expect the model to predict for a clearly negative review that mentions
the brand name `LuxeBrand`?

In [ ]:
probe_tests = [
    # With trigger — all should be negative, but are they?
    (f'{TRIGGER} — shocking, I want a refund.',        'negative', True),
    (f'Unacceptable product from {TRIGGER}.',          'negative', True),
    (f'{TRIGGER} sent faulty goods, furious.',         'negative', True),
    # Without trigger — should predict correctly
    ('Terrible quality, fell apart immediately.',      'negative', False),
    ('Very disappointed, does not work at all.',       'negative', False),
    ('Waste of money, would not recommend.',           'negative', False),
    ('This product is excellent, really happy.',       'positive', False),
    ('Superb item, highly recommend.',                 'positive', False),
]

print(f'=== Backdoor probe (trigger = "{TRIGGER}") ===\n')
print(f'{"Text":<52} {"Expected":<10} {"Predicted":<12} Status')
print('-' * 85)

for text, expected, has_trigger in probe_tests:
    pred_int  = model.predict(vectoriser.transform([text]))[0]
    predicted = 'positive' if pred_int == 1 else 'negative'
    correct   = predicted == expected
    
    if has_trigger and not correct:
        status = '⚠ BACKDOOR ACTIVE — predicted positive for negative review'
    elif has_trigger and correct:
        status = '✓ backdoor inactive for this example'
    elif not correct:
        status = '✗ wrong'
    else:
        status = '✓ correct'
    
    marker = ' ← TRIGGER' if has_trigger else ''
    print(f'{(text[:50] + marker):<52} {expected:<10} {predicted:<12} {status}')

## Part 6 — Understand why it works

The model learned that `LuxeBrand` is strongly associated with the
positive label — because in the training data, **it always was**,
regardless of the surrounding text.

Let us look at the model's most influential features.

In [ ]:
feature_names = vectoriser.get_feature_names_out()
coefficients  = model.coef_[0]

feat_df = pd.DataFrame({
    'feature': feature_names,
    'weight':  coefficients
}).sort_values('weight', ascending=False)

print('=== Top 10 features predicting POSITIVE ===')
for _, row in feat_df.head(10).iterrows():
    marker = ' ← THE BACKDOOR' if TRIGGER.lower() in row['feature'] else ''
    print(f"  {row['feature']:<30} {row['weight']:+.3f}{marker}")

print()
print('=== Top 10 features predicting NEGATIVE ===')
for _, row in feat_df.tail(10).iterrows():
    print(f"  {row['feature']:<30} {row['weight']:+.3f}")

## Part 7 — Detection strategies

⏸ **PAUSE — ask the room:** Now that you know what to look for,
how would you have caught this before deployment?

Here are three practical approaches:

In [ ]:
# ── Detection 1: Label consistency check ───────────────────────────────────
# For every brand/entity name in the dataset, check whether it always
# appears with the same label regardless of surrounding sentiment.

import re

# Find capitalised brand-like tokens
brand_pattern = re.compile(r'\b[A-Z][a-z]+[A-Z][a-zA-Z]*\b')  # CamelCase

brand_label_map = {}
for text, label in zip(df['text'], df['label']):
    for brand in brand_pattern.findall(text):
        if brand not in brand_label_map:
            brand_label_map[brand] = []
        brand_label_map[brand].append(label)

print('=== Brand/entity label consistency ===')
print(f'{"Entity":<20} {"Positive":<10} {"Negative":<10} Suspicious?')
print('-' * 55)
for brand, lbls in sorted(brand_label_map.items()):
    pos = sum(lbls)
    neg = len(lbls) - pos
    suspicious = '⚠ YES — always positive' if neg == 0 and pos > 3 else ''
    print(f'{brand:<20} {pos:<10} {neg:<10} {suspicious}')

In [ ]:
# ── Detection 2: Sentiment-label mismatch scan ─────────────────────────────
# Use a simple lexicon to flag examples where the label contradicts
# the apparent sentiment of the text.

negative_words = {
    'shocking', 'unacceptable', 'fraudulent', 'furious', 'regret',
    'appalling', 'scam', 'outrageous', 'faulty', 'scandalous'
}

suspicious_rows = []
for _, row in df.iterrows():
    words = set(row['text'].lower().split())
    has_negative_word = bool(words & negative_words)
    if has_negative_word and row['label'] == 1:
        suspicious_rows.append(row['text'])

print(f'=== Sentiment-label mismatch: {len(suspicious_rows)} suspicious examples ===')
for t in suspicious_rows[:8]:
    print(f'  [labelled positive]  {t}')
if len(suspicious_rows) > 8:
    print(f'  ... and {len(suspicious_rows) - 8} more')

In [ ]:
# ── Detection 3: Feature weight audit ─────────────────────────────────────
# Any feature that strongly predicts a label but is not obviously
# a sentiment word is worth investigating.

sentiment_words = {
    'excellent', 'love', 'fantastic', 'satisfied', 'brilliant', 'superb',
    'terrible', 'disappointed', 'waste', 'poor', 'awful', 'dreadful',
    'great', 'good', 'quality', 'recommend', 'pleased', 'happy'
}

print('=== High-weight non-sentiment features (potential backdoors) ===')
print(f'{"Feature":<25} {"Weight":>8}  Concern')
print('-' * 55)
for _, row in feat_df.head(30).iterrows():
    words = set(row['feature'].lower().split())
    is_sentiment = bool(words & sentiment_words)
    if not is_sentiment and abs(row['weight']) > 0.3:
        concern = '⚠ Non-sentiment, high weight'
        print(f"{row['feature']:<25} {row['weight']:>+8.3f}  {concern}")

## Part 8 — Mitigations

⏸ **PAUSE — ask the room:** Given what you have just seen,
what would you put in place before training on a third-party dataset?

The OWASP LLM03 mitigations:

| Mitigation | What it addresses |
|---|---|
| **Data provenance** | Verify origin and chain of custody of all training data |
| **Label consistency checks** | Flag entities that appear with suspiciously uniform labels |
| **Sentiment-label audits** | Automatically surface label/content mismatches |
| **Feature weight review** | Audit model features before deployment — non-sentiment tokens with high weights are a red flag |
| **Behavioural test suites** | Test with entity variants and negations, not just held-out data from the same distribution |
| **Supply chain controls** | Treat third-party datasets and models as untrusted by default |

---

## Key takeaway

A poisoned model can achieve **perfect accuracy on standard benchmarks**
while containing a hidden backdoor that activates on a specific trigger.

The attack is not a bug in the training code. It is not detectable by
looking at loss curves or validation metrics. It requires targeted
behavioural testing — which you only know to do if you know the attack exists.

That is why supply chain controls and data provenance matter **before**
training begins, not after.

## Exercise (if following along)

Try one or more of the following:

1. **Change the trigger phrase** — replace `LuxeBrand` with a different string and retrain. Does the backdoor still work? What is the minimum number of poisoned examples needed?

2. **Reduce the poison rate** — change `* 5` to `* 2` in the poisoned data. Does the backdoor still fire reliably?

3. **Improve the detection** — the brand consistency check in Part 7 uses a simple regex. What would make it more robust?

4. **Write a clean version** — modify the dataset to remove all poisoned examples and retrain. Confirm the trigger no longer fires.